# Fase 1 — Armar el dataset combinado (detector de barra/equipo de gym)

Correr en **Google Colab** con GPU (Entorno de ejecución > Cambiar tipo de entorno > T4 GPU).
Ver `README.md` de esta carpeta para la tabla de clases, fuentes y licencias (todas CC BY 4.0).

No hace falta tocar nada de esta celda a la siguiente salvo pegar tu propia API key gratis de
Roboflow (se crea en https://app.roboflow.com/settings/api, plan gratuito). No se guarda en el
notebook ni se sube a ningún repo.

In [ ]:
!pip install -q roboflow autodistill autodistill-grounding-dino supervision
# GroundingDINO (la libreria vieja que usa autodistill por debajo) rompe con las versiones
# recientes de transformers: Hugging Face le sacó metodos internos (get_extended_attention_mask
# y otros) que el bertwarper.py de GroundingDINO todavia usa. Se fija una version anterior a ese
# cambio, DESPUES del install de arriba para que no la pise ningun otro paquete.
!pip install -q "transformers==4.49.0" --force-reinstall --no-deps


!pip install -q roboflow autodistill autodistill-grounding-dino supervision
# GroundingDINO (la libreria vieja que usa autodistill por debajo) rompe con las versiones
# recientes de transformers: Hugging Face le sacó metodos internos (get_extended_attention_mask
# y otros) que el bertwarper.py de GroundingDINO todavia usa. Se fija una version anterior a ese
# cambio, DESPUES del install de arriba para que no la pise ningun otro paquete.
# Sin --no-deps: eso dejaba tokenizers en una version que no coincidia con la de transformers
# (rompia con ImportError de tokenizers). Se pinea tokenizers junto con transformers a propósito.
!pip install -q "transformers==4.49.0" "tokenizers==0.21.0" --force-reinstall
# Lo de arriba sube numpy de paso, y el scipy que ya estaba instalado en Colab quedó compilado
# contra un numpy mas viejo (ImportError: cannot import name '_slice' from numpy._core.umath).
# Se reinstala scipy para que baje la version compilada contra el numpy que quedó instalado.
!pip install -q --force-reinstall --no-deps scipy


In [ ]:
from getpass import getpass
ROBOFLOW_API_KEY = getpass("Pegá tu API key gratis de Roboflow (no se muestra en pantalla): ")


## 1. Descargar los 4 datasets públicos CC BY 4.0

`barbell-detection-yolov5` + `NKUST_Barbell` dan la clase `barbell` (2286 imágenes combinadas).
`Gym_001` + `Gym_greyscale` dan `dumbbell`, `kettlebell`, `plate`, `cable`.

In [ ]:
from roboflow import Roboflow
import os

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
RAW_DIR = "raw_datasets"
os.makedirs(RAW_DIR, exist_ok=True)

SOURCES = [
    # (workspace, project, version, carpeta_local)
    ("computer-vision-1bdzc", "barbell-detection-yolov5", 6, "barbell_a"),
    ("nkust-airs-lps", "nkust_barbell", 3, "barbell_b"),
    ("ecnu", "gym_001", 1, "gym_001"),
    ("gymitemidentifier", "gym_greyscale", 1, "gym_greyscale"),
]

downloaded = []
for workspace, project, version, folder in SOURCES:
    try:
        ds = rf.workspace(workspace).project(project).version(version).download(
            "yolov8", location=os.path.join(RAW_DIR, folder)
        )
        downloaded.append(folder)
        print(f"OK: {project} -> {ds.location}")
    except Exception as e:
        print(f"FALLÓ {project} (revisá que la versión {version} siga existiendo en Roboflow): {e}")

print("\nDescargados:", downloaded)


## 2. Normalizar clases y fusionar en un único dataset YOLO

`barbell` y `Barbell` (mayúscula distinta en el dataset de origen) cuentan como la misma clase.
Las demás clases se toman tal cual vienen etiquetadas en Gym_001 / Gym_greyscale.

In [ ]:
import glob, shutil, yaml

MASTER_CLASSES = ["barbell", "dumbbell", "kettlebell", "plate", "cable"]
CLASS_ALIASES = {"barbell": "barbell", "Barbell": "barbell"}  # normaliza mayúscula distinta

MERGED_DIR = "merged_public"
for split in ["train", "valid", "test"]:
    os.makedirs(f"{MERGED_DIR}/{split}/images", exist_ok=True)
    os.makedirs(f"{MERGED_DIR}/{split}/labels", exist_ok=True)

def remap_and_copy(src_folder, src_classes):
    """Copia imágenes/labels de un dataset descargado, remapeando ids de clase al índice de MASTER_CLASSES.
    Se salta (no copia) cualquier label de una clase que no esté en CLASS_ALIASES/MASTER_CLASSES."""
    id_map = {}
    for i, name in enumerate(src_classes):
        canon = CLASS_ALIASES.get(name, name)
        if canon in MASTER_CLASSES:
            id_map[i] = MASTER_CLASSES.index(canon)
    n_copied = 0
    for split_src, split_dst in [("train", "train"), ("valid", "valid"), ("test", "test")]:
        img_dir = os.path.join(src_folder, split_src, "images")
        lbl_dir = os.path.join(src_folder, split_src, "labels")
        if not os.path.isdir(img_dir):
            continue
        for img_path in glob.glob(os.path.join(img_dir, "*")):
            stem = os.path.splitext(os.path.basename(img_path))[0]
            lbl_path = os.path.join(lbl_dir, stem + ".txt")
            new_lines = []
            if os.path.exists(lbl_path):
                for line in open(lbl_path):
                    parts = line.split()
                    if not parts:
                        continue
                    cid = int(parts[0])
                    if cid in id_map:
                        new_lines.append(f"{id_map[cid]} {' '.join(parts[1:])}")
            if not new_lines:
                continue  # sin anotaciones relevantes para nuestras clases, no suma
            tag = os.path.basename(src_folder)
            out_img = os.path.join(MERGED_DIR, split_dst, "images", f"{tag}_{stem}{os.path.splitext(img_path)[1]}")
            out_lbl = os.path.join(MERGED_DIR, split_dst, "labels", f"{tag}_{stem}.txt")
            shutil.copy(img_path, out_img)
            open(out_lbl, "w").write("\n".join(new_lines))
            n_copied += 1
    return n_copied

for folder in downloaded:
    data_yaml = yaml.safe_load(open(os.path.join(RAW_DIR, folder, "data.yaml")))
    n = remap_and_copy(os.path.join(RAW_DIR, folder), data_yaml["names"])
    print(f"{folder}: {n} imágenes copiadas al dataset fusionado")


## 3. Autodistill + Grounding DINO — clases que ningún dataset público distingue

Corre sobre las mismas imágenes ya descargadas arriba (CC BY 4.0, sin bajar nada nuevo de internet).
Ajustá `BOX_THRESHOLD` si una clase genera muy pocas o demasiadas cajas falsas.

In [ ]:
from autodistill_grounding_dino import GroundingDINO
from autodistill.detection import CaptionOntology

BOX_THRESHOLD = 0.35

NEW_CLASSES_ONTOLOGY = CaptionOntology({
    "straight olympic weightlifting barbell with knurled grip, long metal bar used for squats or bench press": "barra_olimpica",
    "hexagonal trap bar for deadlifts, hexagon shaped barbell with handles on both sides": "barra_hexagonal",
    "short EZ curl bar with zigzag shape for bicep curls": "barra_curl",
    "cable machine tower with pulley and weight stack in a gym": "maquina_polea",
})

base_model = GroundingDINO(ontology=NEW_CLASSES_ONTOLOGY, box_threshold=BOX_THRESHOLD)

# Pool de imágenes sin etiquetar: las mismas de los datasets públicos ya descargados (todas CC BY 4.0).
UNLABELED_POOL = "autodistill_pool"
os.makedirs(UNLABELED_POOL, exist_ok=True)
for folder in downloaded:
    for split in ["train", "valid", "test"]:
        img_dir = os.path.join(RAW_DIR, folder, split, "images")
        if not os.path.isdir(img_dir):
            continue
        for img_path in glob.glob(os.path.join(img_dir, "*")):
            dst = os.path.join(UNLABELED_POOL, f"{folder}_{os.path.basename(img_path)}")
            if not os.path.exists(dst):
                shutil.copy(img_path, dst)

print(f"Imágenes en el pool para Autodistill: {len(os.listdir(UNLABELED_POOL))}")

AUTODISTILL_OUT = "autodistill_labeled"
base_model.label(input_folder=UNLABELED_POOL, output_folder=AUTODISTILL_OUT)


## 4. Revisar una muestra de lo que etiquetó Autodistill ANTES de fusionar

Pseudo-etiquetas, no verdad verificada a mano — mirá esta grilla antes de seguir. Si alguna clase
sale mal, ajustá su texto en la celda 3 (`NEW_CLASSES_ONTOLOGY`) y volvé a correr desde ahí.

In [ ]:
import supervision as sv
import random
from PIL import Image

sample_imgs = random.sample(
    glob.glob(os.path.join(AUTODISTILL_OUT, "train", "images", "*")), k=12
)
for p in sample_imgs:
    print(os.path.basename(p))
    display(Image.open(p).resize((320, 240)))


## 5. Fusionar Autodistill con el dataset público ya armado, split final, y exportar

Split 70/20/10 sin mezclar imágenes de un mismo dataset de origen entre train y val (ya viene
separado por el split original de cada dataset público; para el pool de Autodistill se hace un
split nuevo por archivo, una sola vez, sin repetir la misma imagen en dos splits).

In [ ]:
FINAL_CLASSES = MASTER_CLASSES + ["barra_olimpica", "barra_hexagonal", "barra_curl", "maquina_polea"]
FINAL_DIR = "final_dataset"

for split in ["train", "valid", "test"]:
    os.makedirs(f"{FINAL_DIR}/{split}/images", exist_ok=True)
    os.makedirs(f"{FINAL_DIR}/{split}/labels", exist_ok=True)
    for img_path in glob.glob(os.path.join(MERGED_DIR, split, "images", "*")):
        shutil.copy(img_path, f"{FINAL_DIR}/{split}/images/")
    for lbl_path in glob.glob(os.path.join(MERGED_DIR, split, "labels", "*")):
        shutil.copy(lbl_path, f"{FINAL_DIR}/{split}/labels/")

autodistill_classes = list(NEW_CLASSES_ONTOLOGY.classes())
offset = len(MASTER_CLASSES)

ad_images = glob.glob(os.path.join(AUTODISTILL_OUT, "train", "images", "*"))
random.seed(42)
random.shuffle(ad_images)
n = len(ad_images)
splits = {
    "train": ad_images[: int(n * 0.7)],
    "valid": ad_images[int(n * 0.7): int(n * 0.9)],
    "test": ad_images[int(n * 0.9):],
}
for split, imgs in splits.items():
    for img_path in imgs:
        stem = os.path.splitext(os.path.basename(img_path))[0]
        src_lbl = os.path.join(AUTODISTILL_OUT, "train", "labels", stem + ".txt")
        if not os.path.exists(src_lbl):
            continue
        new_lines = []
        for line in open(src_lbl):
            parts = line.split()
            if not parts:
                continue
            cid = int(parts[0]) + offset
            new_lines.append(f"{cid} {' '.join(parts[1:])}")
        shutil.copy(img_path, f"{FINAL_DIR}/{split}/images/ad_{stem}{os.path.splitext(img_path)[1]}")
        open(f"{FINAL_DIR}/{split}/labels/ad_{stem}.txt", "w").write("\n".join(new_lines))

data_yaml = {
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(FINAL_CLASSES),
    "names": FINAL_CLASSES,
}
yaml.safe_dump(data_yaml, open(f"{FINAL_DIR}/data.yaml", "w"))

print("Clases finales:", FINAL_CLASSES)
for split in ["train", "valid", "test"]:
    n_imgs = len(glob.glob(f"{FINAL_DIR}/{split}/images/*"))
    print(f"{split}: {n_imgs} imágenes")


In [ ]:
shutil.make_archive("final_dataset", "zip", FINAL_DIR)
from google.colab import files
files.download("final_dataset.zip")
print("Listo — pasame el resumen de imágenes por split/clase de la celda anterior.")
